<button data-commandLinker-command="progressivis:cleanup_and_run" data-commandlinker-args='{"index": 1}' href="#" class='progressivis-cleanup-and-run-btn'>Run ProgressiVis</button>

In [ ]:
from ipyprogressivis.widgets.chaining.constructor import Constructor
from ipyprogressivis.widgets.chaining.utils import create_root, get_header
from ipyprogressivis.widgets.chaining.custom import *
# ***************************************************************************************
# WARNING: This cell must only be executed using the 'Run ProgressiVis' button above.
# Do not execute it in any other way, as the result will not be as expected.
# For the same reason do not copy/paste the contents of this cell to execute it elsewhere
# ***************************************************************************************
header = get_header()
display(header.talker)
display(header.backup)
_ = header.constructor
with header.modules_out:
    display(header.board)
with header.widgets_out:
    display(header.manager)
header.talker.labcommand("notebook:hide-cell-code")
%reload_ext ipyprogressivis.magics
create_root(header.backup)

Talker()

BackupWidget()

## root

In [1]:
# do not run this cell
display(header.constructor)
header.constructor.start_scheduler()
header.talker.labcommand('notebook:hide-cell-code')

Constructor(children=(IntProgress(value=0, description='Starting ProgressiVis ...', max=2, style=ProgressStyle…

Starting scheduler
# Scheduler added module(s): ['sink_1', 'variable_1']


## CSV loader

In [2]:
Constructor.widget('CSV loader', 0)

NodeCarrier(children=(HBox(children=(Button(button_style='danger', icon='trash', style=ButtonStyle(), tooltip=…

In [3]:
# progressivis-snippet
from ipyprogressivis.widgets.chaining.custom import register_snippet, SnippetResult
from ipywidgets import Output, VBox, HBox, Dropdown, IntSlider, Label
import plotly.graph_objects as go
import progressivis.core.aio as aio
import numpy as np
import re
from sklearn.decomposition import IncrementalPCA
from sklearn.preprocessing import StandardScaler


@register_snippet
def progressive_incident_pca(input_module, input_slot, columns):

    table = input_module.output[input_slot].data()
    out = Output()

    if not hasattr(progressive_incident_pca, "_state"):

        # 🎨 ColorBrewer Sequential Oranges
        oranges_scale = [
            [0.0, "#feedde"],
            [0.25, "#fdbe85"],
            [0.5, "#fd8d3c"],
            [0.75, "#e6550d"],
            [1.0, "#a63603"]
        ]

        fig = go.FigureWidget(
            go.Scatter(
                x=[], y=[],
                mode="markers",
                marker=dict(
                    size=8,
                    color=[],
                    colorscale=oranges_scale,
                    cmin=1,
                    cmax=5,
                    showscale=True,
                    colorbar=dict(
                        title="Priority",
                        tickvals=[1,2,3,4,5]
                    )
                ),
                text=[]
            )
        )

        # Progress Label
        progress_label = Label(value="Processing: 0%")

        state = {
            "cursor": 0,
            "features": {},
            "state": {},
            "location": {},
            "priority": {},
            "scaler": StandardScaler(),
            "pca": IncrementalPCA(n_components=2),
            "points": {},
            "filters": {
                "state": "All",
                "location": "All",
                "priority": 5
            },
            "states": set(),
            "locations": set(),
            "fig": fig
        }

        progressive_incident_pca._state = state

        # ---------- Widgets ----------
        state_dd = Dropdown(options=["All"], value="All", description="State:")
        location_dd = Dropdown(options=["All"], value="All", description="Location:")
        priority_slider = IntSlider(value=5, min=1, max=5, step=1, description="Max Priority:")

        controls = VBox([
            HBox([state_dd, location_dd]),
            priority_slider,
            progress_label
        ])

        # ---------- Feature extraction ----------
        def extract_features(row):
            def to_num(val, default=0):
                if isinstance(val, str):
                    m = re.search(r"\d+", val)
                    return int(m.group(0)) if m else default
                return val if val is not None else default

            return np.array([
                to_num(row.get("reassignment_count", 0)),
                to_num(row.get("reopen_count", 0)),
                to_num(row.get("sys_mod_count", 0)),
                to_num(row.get("priority", 5)),
                to_num(row.get("impact", 3)),
                to_num(row.get("urgency", 3)),
            ], dtype=float)

        # ---------- Update plot ----------
        def update_plot():
            xs, ys, colors, texts = [], [], [], []

            for inc, (x, y) in state["points"].items():
                st = state["state"].get(inc, "Unknown")
                loc = state["location"].get(inc, "Unknown")
                pr = state["priority"].get(inc, 5)

                if state["filters"]["state"] != "All" and st != state["filters"]["state"]:
                    continue
                if state["filters"]["location"] != "All" and loc != state["filters"]["location"]:
                    continue
                if pr > state["filters"]["priority"]:
                    continue

                xs.append(x)
                ys.append(y)
                colors.append(pr)
                texts.append(f"{inc}<br>{st}<br>{loc}")

            fig.data[0].x = xs
            fig.data[0].y = ys
            fig.data[0].marker.color = colors
            fig.data[0].text = texts

            fig.layout.title = "Progressive PCA of Incidents"
            fig.layout.xaxis.title = "PC1"
            fig.layout.yaxis.title = "PC2"

        # ---------- Widget callbacks ----------
        state_dd.observe(lambda c: (state["filters"].update({"state": c["new"]}), update_plot()), names="value")
        location_dd.observe(lambda c: (state["filters"].update({"location": c["new"]}), update_plot()), names="value")
        priority_slider.observe(lambda c: (state["filters"].update({"priority": c["new"]}), update_plot()), names="value")

        # ---------- Display ----------
        display(out)
        with out:
            display(controls)
            display(fig)

        # ---------- Progressive processing ----------
        async def process_rows():
            batch_size = 20

            while state["cursor"] < table.nrow:

                start = state["cursor"]
                end = min(start + batch_size, table.nrow)

                X_batch = []
                incs = []

                for i in range(start, end):
                    row = table.row(i)
                    inc = row["number"]

                    vec = extract_features(row)
                    state["features"][inc] = vec
                    X_batch.append(vec)
                    incs.append(inc)

                    state["state"][inc] = row.get("incident_state", "Unknown")
                    state["location"][inc] = row.get("location", "Unknown")

                    pr = row.get("priority", 5)
                    if isinstance(pr, str):
                        m = re.search(r"\d+", pr)
                        pr = int(m.group(0)) if m else 5
                    state["priority"][inc] = pr

                    state["states"].add(state["state"][inc])
                    state["locations"].add(state["location"][inc])

                if len(X_batch) >= 2:
                    X_batch = np.array(X_batch)
                    X_scaled = state["scaler"].partial_fit(X_batch).transform(X_batch)
                    state["pca"].partial_fit(X_scaled)
                    X_proj = state["pca"].transform(X_scaled)

                    for inc, p in zip(incs, X_proj):
                        state["points"][inc] = (p[0], p[1])

                state_dd.options = ["All"] + sorted(state["states"])
                location_dd.options = ["All"] + sorted(state["locations"])

                state["cursor"] = end

                # 📊 Update Progress Indicator
                percent = round((state["cursor"] / table.nrow) * 100, 1)
                remaining = table.nrow - state["cursor"]
                total_incidents = len(state["points"])

                progress_label.value = (
                    f"Processed: {percent}% | "
                    f"Rows Used: {state['cursor']} | "
                    f"Remaining: {remaining} | "
                    f"Incidents Indexed: {total_incidents}"
                )

                update_plot()
                await aio.sleep(0.05)

        aio.create_task(process_rows())

    return SnippetResult(
        output_module=input_module,
        output_slot=input_slot,
        widget=out
    )

## Snippet

In [4]:
Constructor.widget('Snippet', 0)

NodeCarrier(children=(HBox(children=(Button(button_style='danger', icon='trash', style=ButtonStyle(), tooltip=…